# Retrieval

LangChain에서 Retrieval은 외부 데이터에서 관련 정보를 찾아 프롬프트에 포함시켜(Context) LLM에 전달하는 역할을 한다. 주요 구성 요소는 다음과 같다.

- **Document Loader**: 다양한 원본 데이터를 LangChain 표준 문서 객체로 변환한다.
- **Text Splitter**: 긴 문서를 작은 청크로 분할해 검색 효율을 높인다.
- **Embedding Model**: 텍스트를 의미 기반 벡터로 변환한다.
- **Vector Store**: 임베딩된 벡터를 저장하고 유사도 기반 검색을 지원한다.
- **Retriever**: 쿼리에 대해 관련 문서를 찾아주는 표준 인터페이스를 제공한다.
- **Retrieval Chain**: 검색된 문서를 LLM에 전달해 답변을 생성하는 체인 구조를 제공한다.

이렇게 각 모듈이 결합되어, 외부 데이터 기반의 효과적인 검색 및 답변 생성이 가능하다.

**환각 Hallucination:**

LLM이 실제 근거 없이 그럴듯해 보이는 정보를 생성하는 현상이다.

**주요 원인**
1. **학습 데이터 한계**
   * 모델이 학습한 데이터에 해당 정보가 없거나 부족할 때 발생한다.
2. **확률적 생성 과정**
   * 토큰 예측 시 언어적 일관성을 우선하다 보니, 사실 여부가 검증되지 않은 내용을 생성한다.
3. **프롬프트 모호성**
   * 지시가 불명확하거나 맥락이 부족하면 모델이 관련 없는 정보를 보충·왜곡한다.

**대표 사례**
* 존재하지 않는 논문·저자명을 인용함.
* 역사적·과학적 사실을 잘못 기술함.
* 실행 불가능하거나 비효율적인 코드 제안.


**완화 방안**

1. **지식 기반 검색 결합**
   * Retrieval-Augmented Generation(RAG) 방식으로 외부 문서·데이터베이스에서 실시간 근거를 가져와 보강한다.
2. **프롬프트 구체화**
   * “출처를 함께 제시해 달라” 등 명시적 요청을 통해 근거 표기를 유도한다.
3. **후처리 검증**
   * 생성 결과를 룰 기반 검증 또는 전문가 리뷰를 통해 교차 확인한다.
4. **모델 파인튜닝 및 앙상블**
   * 도메인 특화 데이터로 추가 학습하거나, 룰 기반 시스템과 결합하여 정확도를 높인다.

In [1]:
%pip install langchain langchain-community langchain-openai langchain-huggingface wikipedia pypdf tavily-python tiktoken faiss-cpu sentence-transformers -Uqqq

Note: you may need to restart the kernel to use updated packages.


## Document

Document는 LangChain 프레임워크에서 다양한 데이터 소스(예: 텍스트 파일, PDF, 웹페이지 등)로부터 불러온 정보를 표준화된 객체로 표현하는 핵심 데이터 구조이다. 이 객체는 언어 모델(LLM)이 외부 데이터를 이해하고 처리할 수 있도록 도와준다.

**Document 객체의 구조**
1. page_content: 문서의 실제 내용을 담고 있는 문자열(str)이다. 예를 들어, 텍스트 파일의 본문이나 PDF의 텍스트 등이 여기에 저장된다.
2. metadata: 문서에 대한 부가 정보를 담는 딕셔너리(dict) 형태의 속성이다. 예를 들어, 파일 경로, 페이지 번호, 작성자, 데이터 출처 등 다양한 메타데이터를 저장할 수 있다.


**Document의 역할과 활용**
1. 표준화된 데이터 구조: 다양한 포맷의 데이터를 일관된 방식으로 표현하여, LLM이 손쉽게 접근하고 활용할 수 있도록 한다.
2. 문서 처리의 기본 단위: LangChain의 문서 로더(Document Loader)는 파일, 웹, 데이터베이스 등 여러 소스에서 데이터를 읽어와 Document 객체로 변환한다.
3. 청크 단위 분할: 대용량 문서는 작은 단위(청크)로 쪼개어 각각의 Document로 저장하고, 검색 및 임베딩 처리에 활용한다.

In [2]:
from langchain_core.documents import Document       # Langchain 표준 문서 단위 객체

doc = Document(
    page_content = '이것은 랭체인의 Document 객체입니다. 모든 데이터소스는 이 Document 객체로 변환됩니다.',
    metadata = {
        'source' : 'durlwjrl',          # 데이터 출처
        'url' : 'https://encore.com',   # 원문 url
        'timestamp': 202608250939       # 수집/ 생성 시간
    }
)


## Document Loader
https://reference.langchain.com/python/langchain_core/document_loaders/


Document Loader는 다양한 데이터 소스에서 데이터를 읽어와 Document 객체로 변환하는 역할을 한다. 예를 들어, PDFLoader, CSVLoader, TextLoader 등 다양한 종류가 존재하며, 각기 다른 파일 형식을 Document 객체로 표준화한다.

Document Loader는 데이터 소스별로 특화된 클래스를 제공하며, 문서를 로드한 후 LangChain에서 사용하는 표준 형식으로 변환해준다.

1. **다양한 데이터 소스 지원**  
   Document Loader는 파일 시스템, 클라우드 스토리지, 데이터베이스, 웹 등 다양한 데이터 소스에서 데이터를 로드할 수 있도록 설계되었다.
   
2. **표준화된 출력 형식**  
   로드된 문서는 LangChain에서 사용하는 `Document` 객체로 변환된다. `Document` 객체는 다음과 같은 필드를 포함한다:
   - `page_content`: 문서 본문 내용
   - `metadata`: 문서와 관련된 메타데이터 (예: 파일 이름, URL, 작성자 등)

3. **플러그인 기반 확장 가능**  
   사용자 정의 데이터 소스 로더를 쉽게 구현하고 LangChain에 통합할 수 있다.

**주요 Document Loader 예시**

| Loader 이름        | 설명                                                              |
|--------------------|-------------------------------------------------------------------|
| `PyPDFLoader`      | PDF 문서를 로드하며 텍스트를 추출해 Document 형식으로 변환한다.     |
| `TextLoader`       | 일반 텍스트 파일을 로드한다.                                      |
| `UnstructuredFileLoader` | 비구조적 데이터를 로드하여 구조화된 텍스트로 변환한다.           |
| `CSVLoader`        | CSV 파일에서 데이터를 로드하며 행(row)을 Document로 처리한다.      |
| `WebBaseLoader`    | 웹 페이지 데이터를 크롤링하여 Document로 로드한다.                |

In [3]:
from langchain_community.document_loaders import WebBaseLoader

url = 'https://n.news.naver.com/article/047/0002526580'

header = {
    # 브라우저 식별 : Windows 에서 chrome으로 접속한 것처럼 보이게 만드는 UA
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

loader = WebBaseLoader(url, header_template=header)     # 로더 객체 생성
docs = loader.load()            # 불러온 웹페이지 -> Document 리스트
docs

C:\Users\UK\AppData\Local\Temp\ipykernel_39048\4129421422.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://n.news.naver.com/article/047/0002526580', 'title': "'두 번 멸종한 동물'의 정체... 인간은 참 오만했다", 'language': 'ko'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\'두 번 멸종한 동물\'의 정체... 인간은 참 오만했다\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문 바로가기\n\n\n\n\n\n\n이전 페이지\n\n\n\n\n\n\n\n\n\n\n오마이뉴스\n\n\n\n\n\n구독\n\n메인 뉴스판에서 오마이뉴스 주요뉴스를 볼 수 있습니다.\n보러가기\n닫기\n\n\n오마이뉴스 언론사 구독 해지되었습니다.\n닫기\n\n\n\n\n\n\n\n\n\n\n주요뉴스\n클립\n이슈\n정치\n경제\n사회\n생활\n세계\n랭킹\n\n\n\nMY\n\n\n뉴스 이용 설정을 할 수 있어요\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n오마이뉴스\n\n\n\nPICK\n안내\n\n\n언론사가 주요기사로선정한 기사입니다.\n언론사별 바로가기\n닫기\n\n\n\n\n\'두 번 멸종한 동물\'의 정체... 인간은 참 오만했다\n\n\n\n\n\n\n\n\n입력\n2026.08.25. 오전 7:26\n\n\n\n기사원문\n \n\n\n\n\n\n\n\n\n\n\n추천\n반응\n\n\n\n\n쏠쏠정보\n0\n\n\n\n\n흥미진진\n0\n\n\n\n\n공감백배\n0\n\n\n\n\n분석탁월\n0\n\n\n\n\n후속강추\n0\n\n\n \n\n\n\n\n댓글\n반응\n\n\n\n\n\n\n\n\n텍스트 음성 변환 서비스 사용하기\n\n\n\n성별\n남성\n여성\n\n\n말하기 속도\n느림\n보통\n빠름\n\n이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.\n본문듣기 시작\n\n닫기\n\n\n \n\n글자 크기 변경하기\n\n글자크

In [4]:
print(len(docs))

doc = docs[0]
print(doc.metadata)
print(doc.metadata['title'])
print(doc.page_content)

1
{'source': 'https://n.news.naver.com/article/047/0002526580', 'title': "'두 번 멸종한 동물'의 정체... 인간은 참 오만했다", 'language': 'ko'}
'두 번 멸종한 동물'의 정체... 인간은 참 오만했다





















'두 번 멸종한 동물'의 정체... 인간은 참 오만했다














본문 바로가기






이전 페이지










오마이뉴스





구독

메인 뉴스판에서 오마이뉴스 주요뉴스를 볼 수 있습니다.
보러가기
닫기


오마이뉴스 언론사 구독 해지되었습니다.
닫기










주요뉴스
클립
이슈
정치
경제
사회
생활
세계
랭킹



MY


뉴스 이용 설정을 할 수 있어요


















오마이뉴스



PICK
안내


언론사가 주요기사로선정한 기사입니다.
언론사별 바로가기
닫기




'두 번 멸종한 동물'의 정체... 인간은 참 오만했다








입력
2026.08.25. 오전 7:26



기사원문
 










추천
반응




쏠쏠정보
0




흥미진진
0




공감백배
0




분석탁월
0




후속강추
0


 




댓글
반응








텍스트 음성 변환 서비스 사용하기



성별
남성
여성


말하기 속도
느림
보통
빠름

이동 통신망을 이용하여 음성을 재생하면 별도의 데이터 통화료가 부과될 수 있습니다.
본문듣기 시작

닫기


 

글자 크기 변경하기

글자크기


가1단계
작게


가2단계
보통


가3단계
크게


가4단계
아주크게


가5단계
최대크게


닫기




SNS 보내기



인쇄하기








[이하늬의 멸종위기동물] 피레네아이백스의 잔혹사황량하고 혹독한 피레네산맥의 깎아지른 절벽, 인간의 발길이 쉽게 닿지 않는 협곡 사이를 바람처럼 누비던 동물이 있었습니다. 65~90cm에 달하는 웅장하고 아름다운 뿔, 험준한 바위 산책로를 마치 평지처럼 유희

In [5]:
!gdown 1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO

Downloading...
From: https://drive.google.com/uc?id=1o7ngiyeJJ-MPLhl0fiCKHViTNNpk6zjO
To: c:\Users\UK\SKN\LLM\05_langchain\02_langchain_component\The_Adventures_of_Tom_Sawyer.pdf

  0%|          | 0.00/2.68M [00:00<?, ?B/s]
 20%|█▉        | 524k/2.68M [00:00<00:00, 2.39MB/s]
 59%|█████▊    | 1.57M/2.68M [00:00<00:00, 5.30MB/s]
100%|██████████| 2.68M/2.68M [00:00<00:00, 7.02MB/s]


In [6]:
from langchain_community.document_loaders import PyPDFLoader    # PDF 를 읽어 Document 로 로드하는 로더

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()    # PDF를 페이지별 Document 리스트로 변환
print(len(docs))    # Document 수

35


In [7]:
print(docs[2].metadata)             # 3 페이지의 메타데아터
print(docs[2].metadata['source'])   # 경로
print(docs[2].metadata['page'])     # 현재 페이지
print(docs[2].metadata['page_label'])   # 사람이 볼수있는 페이지 번호
print(docs[2].page_content)         # 본문

{'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 2, 'page_label': '3'}
The_Adventures_of_Tom_Sawyer.pdf
2
3
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter


### TavilySearchAPIRetriever
https://www.tavily.com/

- `langchain_tavily.TavilySearch`: Agent tool사용버젼. json반환
- `langchain_community.retrievers.TavilySearchAPIRetriever`: 검색기(context확보용) Document객체반환

- 주요 기능
    - 웹 검색(query → 결과 리스트): 키워드로 웹을 검색해서 관련 페이지들을 찾아줌
    - 요약/스니펫 제공: 각 결과에 본문 요약이나 핵심 스니펫을 같이 줘서 LLM이 바로 쓰기 좋음
    - 컨텐츠 추출(include_raw_content 등 옵션): 결과 페이지의 내용을 일부/전체 텍스트로 가져오게 설정 가능
    - 필터링/튜닝 옵션: 검색 결과 개수, 도메인 포함/제외, 최신성(리센시) 같은 옵션으로 결과를 조절 가능
    - RAG 파이프라인에 바로 연결: “검색 → 문서(Document)화 → 벡터화/리랭킹 → 답변” 흐름에서 검색 단계로 많이 사용

In [8]:
!pip install tavily-python -qqq

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

In [11]:
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily_retriever = TavilySearchAPIRetriever(k = 3)

docs = tavily_retriever.invoke('런닝')
docs

[Document(metadata={'title': "'러닝', '런닝' 중 올바른 표기는? (런닝머신? 러닝머신!) : 네이버 블로그", 'source': 'https://m.blog.naver.com/chorduk/223308645264', 'score': 0.7395922, 'id': '2a2619-00', 'images': []}, page_content="우리말샘\n\n\u200b\n\n「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷.\n\n\u200b\n\n「002」 「명사」 『체육』 요트 경기에서, 바람을 등지고 달리는 일.\n\n\u200b\n\n「003」 「명사」 『체육』 스키에서, 미끄러져 내려가는 일.\n\n\u200b\n\n「004」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술.\n\n\u200b\n\n이렇게 나와 있습니다.\n\n\u200b\n\n\u200b\n\n\u200b\n\n음.. 그럼 '런닝'은 뭘까요?\n\n\u200b\n\n'러닝'과 같거나 비슷한 뜻을 가진 단어일까요?\n\n\u200b\n\n'러닝'과 다른 뜻을 가진 단어일까요?\n\n\u200b\n\n아니면 '러닝'을 잘못 표기한 걸까요? \u200b\n\n\u200b\n\n\n\n\u200b\n\n\u200b\n\n\u200b\n\n사전에 '런닝'도 검색해 보도록 하겠습니다.\n\n\u200b\n\n\u200b\n\n\u200b\n\n우리말샘에 '런닝'을 검색해 보면\n\n\n\n우리말샘\n\n\u200b [...] 우리말샘\n\n\u200b\n\n「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷. ⇒규범 표기는 ‘러닝’이다.\n\n\u200b\n\n「002」 「명사」 「북한어」 ‘러닝’의 북한어.\n\n\u200b\n\n「003」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 

In [12]:
for doc in docs:
    print(doc.page_content)

우리말샘

​

「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷.

​

「002」 「명사」 『체육』 요트 경기에서, 바람을 등지고 달리는 일.

​

「003」 「명사」 『체육』 스키에서, 미끄러져 내려가는 일.

​

「004」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술.

​

이렇게 나와 있습니다.

​

​

​

음.. 그럼 '런닝'은 뭘까요?

​

'러닝'과 같거나 비슷한 뜻을 가진 단어일까요?

​

'러닝'과 다른 뜻을 가진 단어일까요?

​

아니면 '러닝'을 잘못 표기한 걸까요? ​

​



​

​

​

사전에 '런닝'도 검색해 보도록 하겠습니다.

​

​

​

우리말샘에 '런닝'을 검색해 보면



우리말샘

​ [...] 우리말샘

​

「001」 「명사」 운동 경기할 때 선수들이 입는 소매 없는 셔츠. 또는 그런 모양의 속옷. ⇒규범 표기는 ‘러닝’이다.

​

「002」 「명사」 「북한어」 ‘러닝’의 북한어.

​

「003」 「명사」 『체육』 외국 프로 레슬링에서, 상대를 잡아 들어 올린 후 앞으로 뛰어가 기둥이나 물체에 상대를 부딪치게 하는 기술. ⇒규범 표기는 ‘러닝’이다.

​

보시다시피 이렇게 나와 있습니다.

​

​

​

음.. 그럼 왜 '런닝'이 아닌 '러닝'일까요? ​

​



​

​

​

이건 '러닝(running)'의 발음 기호를 보면 알 수 있습니다.

​

'러닝(running)'의 발음 기호는 [ˈrʌnɪŋ]입니다.

​

'n'이 하나만 있죠?

​

따라서 '런닝'이 아닌 '러닝'으로 적는 것이 올바른 표기입니다. ​

​



​

​

​

그럼 이만 포스팅 마치도록 하겠습니당😆😆

​

​

​

끝까지 읽어 주셔서 감사합니다🧡​

​



​ [...] 본문 바로가기

# 블로그

## 카테고리 이동 고양이 좋아하는 속기사

검색

'러닝', '런닝' 중 

### Tavily 검색 결과로 Context에 넣고 답변하는 RAG chain


In [13]:
from langchain_core.prompts import PromptTemplate          
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

tavily_retriever = TavilySearchAPIRetriever(k= 3)   # 검색결과 상위 3개
prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser()

def format_docs(docs: list[Document]) -> str:
    return '\n\n'.join(doc.page_content for doc in docs)

tavily_chain = tavily_retriever | format_docs

chain = (
    {'question': RunnablePassthrough(), 'context': tavily_chain} | prompt | llm | output_parser
)

chain.invoke('8월 말 독산역 인기 맛집?')

'제공된 정보 기준으로 **8월 말 독산역 인기 맛집**을 고르면 다음과 같습니다.\n\n1. **화로구이 독산점** – 고기구이  \n   - 블로그 리뷰 434개, 방문자 리뷰 1,280개로 가장 눈에 띄는 인기 매장입니다.  \n   - 삼겹살·돼지갈비·한우·냉면 메뉴가 있고, **24시간 연중무휴**라 늦은 시간 방문에도 편합니다.  \n   - 독산역 1번 출구에서 도보 약 3분 거리입니다.  \n   - 단체회식이나 여럿이 먹는 식사에 추천합니다.\n\n2. **기주짬뽕 본점** – 중식  \n   - 짬뽕, 해물짬뽕, 짜장면, 간짜장, 탕수육을 찾는 분에게 적합합니다.  \n   - 제공된 태그와 키워드상 **짬뽕해장·간짜장·미니탕수육**을 원하는 경우 잘 맞습니다.  \n   - 홈플러스 뒤편 현대지식산업센터에 있어 독산역 인근 직장인 점심 식사로도 좋아 보입니다.\n\n3. **진영면옥** – 냉면·곰탕  \n   - 평양냉면, 비빔냉면, 녹두전, 수육을 판매하는 조용하고 정갈한 분위기의 식당입니다.  \n   - 8월 말처럼 더운 시기에는 냉면 메뉴를 우선 고려하기 좋습니다.  \n   - 혼자 방문하거나 조용히 식사하려는 경우 추천합니다.  \n   - 일요일은 휴무이며, 평일에는 15:00~17:00 브레이크타임이 있습니다.\n\n4. **궁전산들애** – 한정식  \n   - 방문자 리뷰 974개로 인지도가 높고, 간장게장·불고기전골·옥돔정식·제육정식 등이 있습니다.  \n   - 가족 외식, 특별한 날, 단체 모임에 적합합니다.  \n   - 영업시간은 매일 11:00~23:00입니다.\n\n5. **팔곱집** – 곱창·막창·양  \n   - 곱창, 막창, 양을 좋아하거나 술자리·회식 장소를 찾을 때 고려할 만합니다.  \n   - 독산역 인근 주택가에 있으며, 별도 주차장은 없는 것으로 안내되어 있습니다.\n\n**한 곳만 고른다면:**  \n- 인기와 접근성: **화로구이 독산점**  \n- 8월 말 시원한 식사: **진영면옥** 

## Embedding Model
- openai
- setence-transformer(huggingface)

In [14]:
from langchain_openai import OpenAIEmbeddings
import pandas as pd

embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)
print(len(emb_vec))
print(emb_vec[:3])

pd.Series(emb_vec, name= 'embedding')

1536
[-0.0273590087890625, -0.00527191162109375, 0.005352020263671875]


0      -0.027359
1      -0.005272
2       0.005352
3      -0.049652
4       0.026611
          ...   
1531    0.022980
1532   -0.026749
1533   -0.001171
1534    0.036957
1535   -0.023468
Name: embedding, Length: 1536, dtype: float64

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model= 'sentence-transformers/all-MiniLM-L6-v2')
text = '철수는 골든리트리버를 키우고 있습니다.'

emb_vec = embeddings.embed_query(text)
print(len(emb_vec))
print(emb_vec[:3])

pd.Series(emb_vec, name= 'embedding')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\UK\SKN\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\UK\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384
[0.01102148275822401, 0.06504976749420166, 0.04818838834762573]


0      0.011021
1      0.065050
2      0.048188
3     -0.068704
4      0.014493
         ...   
379    0.067011
380    0.011610
381    0.025551
382   -0.018868
383    0.005621
Name: embedding, Length: 384, dtype: float64

### FAISS

- **공식 문서**: https://faiss.ai/
- **Github**: https://github.com/facebookresearch/faiss

**Faiss(Vector Search Library)**는 Facebook AI Research에서 개발한 **효율적인 벡터 검색 및 밀집 벡터 인덱싱 라이브러리**이다. 대규모 데이터에서 **빠른 유사도 검색과 군집화**를 수행하는 데 최적화되어 있다. 주로 문서 검색, 추천 시스템, 이미지 검색, NLP 모델에서 벡터 임베딩 처리를 지원한다.

**주요 특징**
1. **효율적인 유사도 검색**
   - `k-NN (k-Nearest Neighbors)`를 기반으로 벡터 간 유사도(예: 코사인 유사도, L2 거리)를 계산한다.
   - CPU/GPU 모두 지원하여 대규모 데이터에서도 빠르게 처리 가능하다.

2. **고성능 인덱싱**
   - 다양한 **인덱싱 알고리즘**(Flat, IVF, HNSW, PQ 등)을 지원하여 정확도와 속도 간 균형을 맞출 수 있다.
   - 데이터가 커질수록 효율적으로 검색 성능을 발휘하도록 설계되었다.

3. **확장성**
   - 수억 개의 벡터에서도 성능을 유지하도록 설계되었으며, GPU 병렬 처리를 통해 성능을 극대화한다.

4. **유연성**
   - Python과 C++ API를 제공하며, Scikit-learn이나 PyTorch와 같은 다른 라이브러리와 통합하여 사용 가능하다.

**Faiss의 기본 인덱스 유형**
1. **Flat Index**
   - 모든 벡터를 저장하고 전체 탐색(Brute-Force)을 수행.
   - 정확도가 높지만 대규모 데이터에서는 속도가 느릴 수 있다.

2. **IVF (Inverted File Index)**
   - 벡터를 클러스터링하여 데이터 양을 줄이고 탐색 속도를 높임.
   - 대규모 데이터에서 적합하며, 정확도와 속도 조절 가능.

3. **PQ (Product Quantization)**
   - 벡터를 압축하여 메모리 사용량을 줄이고, 빠른 근사 유사도 검색 수행.

4. **HNSW (Hierarchical Navigable Small World Graphs)**
   - 그래프 기반 알고리즘으로 매우 빠른 근사 유사도 검색 가능.


**Faiss의 주요 사용 사례**
1. **문서 검색**
   - 문서를 벡터로 변환한 후 가장 관련 있는 문서를 검색.
   - NLP 모델의 임베딩과 결합하여 사용.

2. **이미지 검색**
   - 이미지 특징 벡터를 사용하여 비슷한 이미지를 검색.

3. **추천 시스템**
   - 사용자의 행동이나 관심사를 벡터화하여 추천 품목 생성.

4. **클러스터링**
   - 벡터 데이터를 군집화하여 데이터의 구조를 분석.

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
import numpy as np

loader = PyPDFLoader('The_Adventures_of_Tom_Sawyer.pdf')
docs = loader.load()
page_contents = [doc.page_content for doc in docs]

embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
emb_vecs = embeddings.embed_documents(page_contents)    # 페이지별 컨텐츠 임베딩 -> 벡터 리스트

np.array(emb_vecs).shape    # (페이지수, 임베딩 차원수)

(35, 1536)

In [19]:
# FAISS 벡터스토어를 이용해 문서들을 임베딩해 로컬에 저장
from langchain_community.vectorstores import FAISS  # FAISS 기반

vector_db = FAISS.from_documents(docs, embeddings)  # docs 를 임베딩해서 FAISS 인덱스 생성
vector_db.save_local('./db/faiss')      # 로컬 경로에 FAISS 인덱스/ 메타데이터 저장


In [20]:
# 로컬에 저장해놓은 FAISS 벡터스토어 로드
vector_db = FAISS.load_local(
    './db/faiss',   # 경로
    embeddings,     # 로드시 사용할 임베딩 모델
    allow_dangerous_deserialization= True   # 신뢰된 파일만 사용(pickle 역직렬화 허용)
)

In [21]:
search_result = vector_db.similarity_search(
    query = '학교 선생님이 아끼는 해부학 책은 누가 찢었는가?',  # 쿼리 : 한글
    k= 4    # 상위 4개 Document
)

search_result   # list[Document]

[Document(id='64367a02-9e05-4581-90b8-ebe91ac4ad29', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 7.0 dla programu Word', 'creationdate': '2006-08-26T00:50:00+02:00', 'author': 'GOLDEN', 'company': 'c', 'title': 'Microsoft Word - 1', 'moddate': '2021-01-27T15:00:11+01:00', 'source': 'The_Adventures_of_Tom_Sawyer.pdf', 'total_pages': 35, 'page': 15, 'page_label': '16'}, page_content='talking about it. Becky wanted to talk to Tom, but he \ndidn’t look at her. \nThen Tom talked to Amy. Becky watched him and she \nwas angry. She said to her friends, “I’m going to have an \nadventure day. You can come on my adventure.” But she \ndidn’t ask Tom. \nLater in the morning, Tom ta lked to Amy again. Becky \ntalked to her friend Alfred and looked at a picture-book \nwith him. Tom watched them and he was angry with \nBecky. \nIn the afternoon, Tom waited for Becky at the school \nfence. He said, “I’m sorry.” \nBut Beck

In [22]:
for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f'{i}번째 {doc.metadata['page_label']} page: {doc.page_content}')


1번째 16 page: talking about it. Becky wanted to talk to Tom, but he 
didn’t look at her. 
Then Tom talked to Amy. Becky watched him and she 
was angry. She said to her friends, “I’m going to have an 
adventure day. You can come on my adventure.” But she 
didn’t ask Tom. 
Later in the morning, Tom ta lked to Amy again. Becky 
talked to her friend Alfred and looked at a picture-book 
with him. Tom watched them and he was angry with 
Becky. 
In the afternoon, Tom waited for Becky at the school 
fence. He said, “I’m sorry.” 
But Becky didn’t listen to him. She walked into the 
school room. The teacher’s new book was on his table. 
This book wasn’t for children, but Becky wanted to look 
at it. She opened the book quietly and looked at the 
pictures. 
Suddenly, Tom came into the room. Becky was 
surprised. She closed the book quickly, and it tore. Becky 
was angry with Tom and quickly went out of the room. 
Then the children and the teacher came into the room 
and went to their places. The t

### VectorStoreRetriever

리트리버는 벡터DB의 검색 기능을 표준화하고 추상화하여 LangChain 생태계에서 재사용성을 높이는 어댑터(Adapter) 역할을 수행한다.

벡터 저장소를 **`Retriever`라는 표준 인터페이스(Runnable)로 변환**한 뒤 실행하는 방식이다.

단순 유사도 검색뿐만 아니라, `search_type` 설정을 통해 **MMR(다양성 확보), 임계값 필터링(score_threshold)** 등 고급 검색 로직을 쉽게 적용할 수 있다.

**LCEL(LangChain Expression Language)** 파이프라인(`chain = retriever | llm`)에 즉시 통합 가능하다. 코드 수정 없이 검색 알고리즘만 교체하기 쉽다.

In [23]:
# VectorStore 를 Retriever 인터페이스 변환
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 3
    }
)

search_result = retriever.invoke('마을 무덤의 남자는 누가 죽였는가?')

for i, doc in enumerate(search_result, 1):
    # 사람이 보는 페이지 : 본문 내용
    print(f'{i}번째 {doc.metadata['page_label']} page: {doc.page_content}')


1번째 19 page: A man asked him, “Where were you on the night of  
June 17th?” 
“I was in the graveyard,” Tom answered. 
“Did you see any people there?” the man asked: 
“Yes. Injun Joe, the doctor, and Muff Potter were there. 
They didn’t see me because I was behind some big trees.” 
“What did you see?” the man asked. 
“Injun Joe and the doctor talked angrily,” Tom 
answered. “Then Injun Joe killed the doctor with his knife. 
Muff Potter didn’t do it.” 
The people at the trial were surprised. Injun Joe quickly 
went out of the building. 
Tom and Huck were very afraid. Tom said, “Now Injun 
Joe knows about us. He can kill us, too.” 
Many people wanted to hear about the boys’ adventure 
in the graveyard. Tom liked talking about it. He was 
happy, too, because he helped Muff Potter. But he didn’t 
sleep well because he was afraid of Injun Joe. 
 
Chapter 7    Injun Joe’s Treasure 
 
One Saturday afternoon, Tom wanted to have an adventure 
because he didn’t want to think about Injun Joe. He w

In [24]:
# 벡터 검색(Retriever) 결과를 Context 에 넣고, PDF 기반 RAG 답변을 생성하는 코드
retriever = vector_db.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 3
    }
)




prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

llm = init_chat_model('openai:gpt-5.6-luna')
output_parser = StrOutputParser()


chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs } | prompt | llm | output_parser
)


# 음식내용 조회 RAG
- 데이터셋 : fine_food_reviews_1k.csv
- 벡터 DB 구성
- Retriever + llm 체인으로 리뷰 조회하는 기능

In [28]:
import pandas as pd

df = pd.read_csv('fine_food_reviews_1k.csv')
data = df['Text'].to_list()

In [29]:
vector_store = FAISS.from_texts(data, embeddings)



In [30]:
# Retriever Chain 요소 구성 (코사인 유사도, 상위 10개 리뷰 검색)
retriever = vector_store.as_retriever(
    search_type = 'similarity', # 검색 방식 : 코사인 유사도
    search_kwargs ={            # 검색 파라미터 묶음
        'k': 10
    }
)

# prompt Chain 요소 구성 (context, question 입력받음)
prompt = PromptTemplate.from_template('''
        사용자의 질문에 Context 기반으로 답변하세요. 모르는 내용은 모른다고 답변하세요.    
        Context : {context}
        Question : {question}
    ''')

# llm/ 문자열 반환 parser 생성
llm = init_chat_model('openai:gpt-5.6-luna')

output_parser = StrOutputParser()


# 체인구성 : question은 입력값 그대로 전달, context 는 검색 + 문서 합치기 | 최종프로젝스 완성 | LLM 호출
chain = (
    {'question': RunnablePassthrough(), 'context': retriever | format_docs } | prompt | llm | output_parser
)

# fresh fruit 관련 답변 출력
print(chain.invoke('fresh fruit'))

신선한 과일에 대한 내용은 말레이시아에서 먹어 본 **람부탄(rambutan)** 후기입니다. 겉은 부드러운 가시가 있고 속은 즙이 많은 과일로, 약간 멍든 상태로 도착했지만 전반적으로 상태가 좋았습니다. 냉장 보관 시 토요일부터 목요일까지 갈변이 거의 없었으며, 4파운드로 약 40명이 충분히 맛볼 수 있었다고 합니다.
